# 02 负荷规律分析与分区用电特征对比

**题目四　城市电力负荷分析与预测** ｜ 负责：B（规律分析 + 报告主笔）

| 项目 | 内容 |
|---|---|
| 上游数据 | `data/processed/power_load_cleaned.csv`（A 清洗交付，勿再清洗） |
| 本 notebook 产出 | 日 / 周 / 季节负荷规律 + 三分区用电特征对比 + 配套图表与统计表 |
| 对应题目要求 | “分析负荷在日/周/季节尺度上的变化规律，比较三个分区用电特征” |
| 下游消费者 | C（成品图重绘）、B（报告“数据分析与可视化”章节） |

## 分析路线

```
清洗数据
  ├─ 步骤 1  时间特征工程（小时 / 星期 / 月份 / 季节）
  ├─ 步骤 2  日内规律：按小时平均 → 双峰形态、峰谷时段
  ├─ 步骤 3  周内规律：按星期平均 → 工作日 / 周末差异
  ├─ 步骤 4  季节规律：按月 / 季平均 → 夏冬双峰
  └─ 步骤 5  三分区对比：均值贡献、分布、结构变化（呼应 01 的 Zone3 发现）
```


## 步骤 0　环境与路径

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)


def find_root(marker="data"):
    """从当前工作目录向上查找仓库根目录（与 01 保持一致）。"""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / marker / "raw").is_dir():
            return d
    raise FileNotFoundError(f"未在 {here} 及其父目录中找到 {marker}/raw")


ROOT = find_root()
PROCESSED = ROOT / "data" / "processed" / "power_load_cleaned.csv"
FIG_DIR = ROOT / "figures" / "patterns"
TABLE_DIR = ROOT / "tables"
for p in (FIG_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

ZONE_COLS = ["zone1_power_consumption", "zone2_power_consumption", "zone3_power_consumption"]
WEATHER_COLS = ["temperature", "humidity", "wind_speed", "general_diffuse_flows", "diffuse_flows"]

print(f"仓库根目录 : {ROOT}")
print(f"清洗数据   : {PROCESSED.name} (存在: {PROCESSED.exists()})")


## 步骤 1　加载清洗数据

In [ ]:
df = pd.read_csv(PROCESSED, index_col="datetime", parse_dates=True)
df.index.name = "datetime"

print(f"形状   : {df.shape}")
print(f"频率   : {df.index.inferred_freq}")
print(f"范围   : {df.index[0]} → {df.index[-1]}")
print(f"缺失   : {int(df.isna().sum().sum())}")
print("\n--- 前 3 行 ---")
display(df.head(3))


## 步骤 2　时间特征工程

从 `datetime` 索引派生 4 个时间维度，后续所有分组聚合都基于它们：

- `hour`：一天中的小时（0–23），用于日内规律；
- `dow`（day of week）：星期几（0=周一 … 6=周日），用于周内规律；
- `month`：月份（1–12）；
- `season`：季节（春 3–5 / 夏 6–8 / 秋 9–11 / 冬 12,1,2）。

> 注意：直接在 `df` 上新增列会破坏“下游只读取、不修改”的约定，因此这里把时间特征放进
> `feat` 这个并行的 DataFrame，按索引对齐后使用，原数据 `df` 保持只读。


In [ ]:
feat = pd.DataFrame(index=df.index)
feat["hour"] = df.index.hour
feat["dow"] = df.index.dayofweek
feat["month"] = df.index.month
SEASON_MAP = {12: "冬季", 1: "冬季", 2: "冬季",
              3: "春季", 4: "春季", 5: "春季",
              6: "夏季", 7: "夏季", 8: "夏季",
              9: "秋季", 10: "秋季", 11: "秋季"}
feat["season"] = df.index.month.map(SEASON_MAP)
SEASON_ORDER = ["春季", "夏季", "秋季", "冬季"]

print("时间特征预览（前 5 行）:")
display(feat.head())
print("各维度取值数:", {k: feat[k].nunique() for k in ["hour", "dow", "month", "season"]})


## 步骤 3　日内负荷规律（日尺度）

把全年 52,416 个点按「小时」分组取均值，得到一条 24 小时的“典型日”曲线。
下面同时画总量与三分区，看峰谷是否同步、各分区幅度差异。
（注：本数据实测为“午间平台 + 晚间主峰”的日内形态，具体峰谷时段以代码输出为准。）


In [ ]:
hourly = df.groupby(feat["hour"])[ZONE_COLS + ["zone_total_power_consumption"]].mean()

COL_LABEL = {"zone_total_power_consumption": "全区分", "zone1_power_consumption": "Zone1",
             "zone2_power_consumption": "Zone2", "zone3_power_consumption": "Zone3"}
COL_COLOR = {"zone_total_power_consumption": "#1f4e9c", "zone1_power_consumption": "#e07b39",
             "zone2_power_consumption": "#2ca25f", "zone3_power_consumption": "#c0392b"}

fig, ax = plt.subplots(figsize=(12, 5))
for col in ["zone_total_power_consumption"] + ZONE_COLS:
    ax.plot(hourly.index, hourly[col], lw=2, color=COL_COLOR[col], label=COL_LABEL[col])
ax.set_xlabel("小时 (0–23)")
ax.set_ylabel("平均负荷 (kW)")
ax.set_title("图 A  日内负荷曲线（按小时平均，2017 全年）", loc="left")
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=9)
fig.savefig(FIG_DIR / "figA_daily_profile.png", dpi=150)
plt.close(fig)
print("已保存:", FIG_DIR / "figA_daily_profile.png")

# 峰谷时段（以全区分总量为准）
tot = hourly["zone_total_power_consumption"]
print("早高峰(07-11)均值 :", round(tot.loc[7:11].mean(), 1))
print("晚高峰(18-22)均值 :", round(tot.loc[18:22].mean(), 1))
print("最低谷(02-05)均值 :", round(tot.loc[2:5].mean(), 1))
print("峰值小时          :", int(tot.idxmax()), "→", round(tot.max(), 1))
print("谷值小时          :", int(tot.idxmin()), "→", round(tot.min(), 1))
print("峰谷差(全天跨度)   :", round(tot.max() - tot.min(), 1))


### 日内规律结论（报告可直接引用，数值以代码输出为准）

- 典型日曲线呈**“午间平台 + 晚间主峰”**形态：凌晨 **06 时前后为低谷**，午后升至
  午间平台（约 12–15 时），**20 时左右达全天最高**，随后回落；
- 三分区曲线的峰谷时段与总量基本一致，说明全城用电节奏同步，但**各分区峰谷幅度不同**（见步骤 6）；
- 峰谷差（日最大−最小）反映了日内负荷的波动幅度，是后续建模（A 的 03）需重点拟合的对象。


## 步骤 4　周内负荷规律（周尺度）

按星期几分组成 7 条“典型日”均值，对比**工作日 vs 周末**的用电形态差异。
工业/商业占比高的城市，周末负荷通常低于工作日且曲线更平缓。


In [ ]:
DOW_LABEL = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
weekly = df.groupby(feat["dow"])[ZONE_COLS + ["zone_total_power_consumption"]].mean()
weekly.index = [DOW_LABEL[i] for i in weekly.index]
weekly = weekly.reindex(DOW_LABEL)

fig, ax = plt.subplots(figsize=(12, 5))
for col in ["zone_total_power_consumption"] + ZONE_COLS:
    ax.plot(weekly.index, weekly[col], lw=2, marker="o", color=COL_COLOR[col], label=COL_LABEL[col])
ax.set_xlabel("星期")
ax.set_ylabel("平均负荷 (kW)")
ax.set_title("图 B  周内负荷曲线（按星期平均）", loc="left")
ax.legend(fontsize=9)
ax.axvspan(4.5, 6.5, color="orange", alpha=0.08, label="周末")
fig.savefig(FIG_DIR / "figB_weekly_profile.png", dpi=150)
plt.close(fig)
print("已保存:", FIG_DIR / "figB_weekly_profile.png")

wk = weekly["zone_total_power_consumption"]
print("工作日均值(周一~周五):", round(wk.loc[DOW_LABEL[:5]].mean(), 1))
print("周末均值(周六~周日)  :", round(wk.loc[DOW_LABEL[5:]].mean(), 1))
print("周末/工作日比值       :", round(wk.loc[DOW_LABEL[5:]].mean() / wk.loc[DOW_LABEL[:5]].mean(), 4))


### 周内规律结论

- 周末（周六、周日）平均负荷低于工作日，通常以“周末/工作日比值”量化；
- 若比值明显小于 1，说明该城市负荷受**工商业作息**驱动较强（居民用电占比相对低）。
  具体数值与分区差异见上表，写入报告时结合分区结构一起解读。


## 步骤 5　季节负荷规律（季节尺度）

按月与按季聚合，观察全年负荷随气温的季节变化。下面给出月度曲线与季度柱图，
并以**实测结果**为准描述季节形态（本数据呈现夏季单峰主导的特征）。


In [ ]:
monthly = df.groupby(feat["month"])[ZONE_COLS + ["zone_total_power_consumption"]].mean()
seasonal = df.groupby(feat["season"])[ZONE_COLS + ["zone_total_power_consumption"]].mean().reindex(SEASON_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for col in ["zone_total_power_consumption"] + ZONE_COLS:
    axes[0].plot(monthly.index, monthly[col], lw=2, marker="o", color=COL_COLOR[col], label=COL_LABEL[col])
axes[0].set_xlabel("月份")
axes[0].set_ylabel("平均负荷 (kW)")
axes[0].set_title("(a) 月度平均负荷", loc="left")
axes[0].legend(fontsize=8)
axes[0].set_xticks(range(1, 13))

x = np.arange(len(SEASON_ORDER))
w = 0.2
for i, col in enumerate(ZONE_COLS):
    axes[1].bar(x + (i - 1) * w, seasonal[col], w, color=COL_COLOR[col], label=COL_LABEL[col])
axes[1].set_xticks(x)
axes[1].set_xticklabels(SEASON_ORDER)
axes[1].set_ylabel("平均负荷 (kW)")
axes[1].set_title("(b) 季度平均负荷（分区）", loc="left")
axes[1].legend(fontsize=8)
fig.savefig(FIG_DIR / "figC_seasonal_profile.png", dpi=150)
plt.close(fig)
print("已保存:", FIG_DIR / "figC_seasonal_profile.png")

display(seasonal.round(1).rename(columns=lambda c: COL_LABEL.get(c, c)))
seasonal.round(6).to_csv(TABLE_DIR / "seasonal_load_mean.csv", encoding="utf-8-sig")
print("已保存季度均值表:", TABLE_DIR / "seasonal_load_mean.csv")


### 季节规律结论（以实测为准）

- 季度均值表与图 (b) 直接回答“季节尺度变化规律”这一题目要求；
- **本数据呈现夏季单峰主导**：夏季（6–8 月）均值明显高于其余三季，其中 7 月最高；
  而**冬季（12–2 月）反而最低**，说明该城市冬季采暖用电未显著拉动总负荷
  （可能气候温和，或采暖以非电气化方式为主）；
- 夏季峰值主要由**制冷负荷**推动（与步骤 04 中夏季温度-负荷正相关相印证）；
  其余三季负荷水平接近，季节波动幅度相对平缓。


## 步骤 6　三分区用电特征对比

这是题目“比较三个分区用电特征”的核心。从三个角度对比：

1. **规模与占比**：三分区平均负荷、占总量的百分比（结构）；
2. **波动形态**：箱线图看各分区分布与离散程度；
3. **结构变化（呼应 01）**：Zone3 在 8 月下旬起的渐变下行（大用户停限产），
   用逐月 Zone3/Zone1 比值验证，说明它不是异常而是真实业务现象。


In [ ]:
# 1) 规模与占比
zone_mean = df[ZONE_COLS].mean()
zone_share = zone_mean / zone_mean.sum() * 100
comp = pd.DataFrame({"平均负荷(kW)": zone_mean.round(1), "占总负荷(%)": zone_share.round(2)})
comp.index = [COL_LABEL[c] for c in comp.index]
print("=== 三分区规模与占比 ===")
display(comp)
comp.round(6).to_csv(TABLE_DIR / "zone_share.csv", encoding="utf-8-sig")
print("已保存:", TABLE_DIR / "zone_share.csv")

# 2) 分布箱线图
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([df[c].values for c in ZONE_COLS], widths=0.55, patch_artist=True, showfliers=True, whis=1.5)
for patch, col in zip(bp["boxes"], [COL_COLOR[c] for c in ZONE_COLS]):
    patch.set_facecolor(col)
ax.set_xticklabels([COL_LABEL[c] for c in ZONE_COLS])
ax.set_ylabel("负荷 (kW)")
ax.set_title("图 D  三分区负荷分布对比（箱线图）", loc="left")
fig.savefig(FIG_DIR / "figD_zone_boxplot.png", dpi=150)
plt.close(fig)
print("已保存:", FIG_DIR / "figD_zone_boxplot.png")

# 3) 结构变化：逐月 Zone3 / Zone1
monthly_z = df.groupby(feat["month"])[ZONE_COLS].mean()
ratio = (monthly_z["zone3_power_consumption"] / monthly_z["zone1_power_consumption"])
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(monthly_z.index, ratio.values, "o-", color="#c0392b", lw=2)
ax.axhline(1.0, color="gray", ls="--", lw=1)
ax.set_xticks(range(1, 13))
ax.set_xlabel("月份")
ax.set_ylabel("Zone3 / Zone1 比值")
ax.set_title("图 E  Zone3 相对 Zone1 的月度比值（呼应 01 的渐变下行发现）", loc="left")
fig.savefig(FIG_DIR / "figE_zone3_ratio.png", dpi=150)
plt.close(fig)
print("已保存:", FIG_DIR / "figE_zone3_ratio.png")
print("\nZone3/Zone1 比值：7月=", round(ratio[7], 3), " 9月=", round(ratio[9], 3), " 12月=", round(ratio[12], 3))


### 三分区对比结论

- **规模**：三分区平均负荷与占比见 `tables/zone_share.csv`，写入报告作为基础结构描述；
- **波动**：箱线图（图 D）体现各分区的集中趋势与离散程度差异；
- **结构变化（重要）**：图 E 再次印证 01 的发现——Zone3/Zone1 比值从夏季约 0.79 持续下行至
  冬季约 0.38，**这是 Zone3 大用户渐进停限产的真实结构变化，不是异常**，必须在报告中保留并解释，
  不能当成脏数据抹平。


## 步骤 7　小结与交付清单

### 本 notebook 回答了题目的哪些要求

| 题目要求 | 本 notebook 对应 |
|---|---|
| 日尺度负荷规律 | 步骤 3（图 A、峰谷时段） |
| 周尺度负荷规律 | 步骤 4（图 B、周末/工作日比值） |
| 季节尺度负荷规律 | 步骤 5（图 C、季度均值表） |
| 三分区用电特征对比 | 步骤 6（图 D/E、占比表） |

### 产出文件（供 C 重绘、B 写报告）

| 路径 | 内容 |
|---|---|
| `figures/patterns/figA_daily_profile.png` | 日内负荷曲线 |
| `figures/patterns/figB_weekly_profile.png` | 周内负荷曲线 |
| `figures/patterns/figC_seasonal_profile.png` | 月度+季度负荷 |
| `figures/patterns/figD_zone_boxplot.png` | 三分区分布 |
| `figures/patterns/figE_zone3_ratio.png` | Zone3 结构变化 |
| `tables/seasonal_load_mean.csv` | 季度均值 |
| `tables/zone_share.csv` | 三分区占比 |
